# Qwen accuracy test on Colab
Runtime → Change runtime type → **GPU** (A100/L4 recommended for 27b; use T4 with `qwen3:14b` or `qwen3:8b`).

In [ ]:
!apt-get -qq install -y zstd pciutils > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1
!nvidia-smi | head -15

In [ ]:
import subprocess, time, os
subprocess.Popen(['ollama','serve'], stdout=open('/content/ollama.log','w'), stderr=subprocess.STDOUT)
time.sleep(6)
MODEL = 'qwen3.5:27b'   # <-- exact tag of the instructor's model (fallback: 'qwen3:14b')
!ollama pull {MODEL}
os.environ['OLLAMA_MODEL'] = MODEL

Upload the project zip (the folder containing `main.py` and `source/`).

In [ ]:
from google.colab import files
up = files.upload()
name = list(up)[0]
!rm -rf /content/proj && mkdir /content/proj && unzip -q -o "{name}" -d /content/proj
!find /content/proj -maxdepth 2 | head

In [ ]:
# Full test: 10 families x N problems (alternating Persian/English). Adjust --per-family.
%cd /content/proj/hybrid_agent
!pip -q install requests
!python source/eval/run_eval.py --per-family 10 --seed 1 --out /content/eval_results.json 2>/dev/null

In [ ]:
# Repeat with other seeds to measure generalisation / variance
!python source/eval/run_eval.py --per-family 10 --seed 7 --out /content/eval_results2.json 2>/dev/null | tail -20

In [ ]:
# Look at failures
import json
rows = json.load(open('/content/eval_results.json'))
for r in rows:
    if not r['ok']:
        print(r['id'], '| expected:', r['expected'], '| got:', r['got'])
        print(r['problem'], '\n')

In [ ]:
# Test the real CLI on the public-style input
import json
json.dump({'id':'t1','problem':'در یک کلاس ۴۰ دانشجو وجود دارد: ۲۴ نفر Python و ۱۸ نفر Java می‌خوانند و ۱۰ نفر هر دو را. چند نفر هیچ‌کدام را نمی‌خوانند؟\nفقط عدد صحیح را برگردانید.'}, open('in.json','w'), ensure_ascii=False)
!python main.py --input in.json --output out.json 2>/dev/null; cat out.json